# D2_02 t-test와 A/B 테스트

> **오늘 질문:** 고장 난 기록은 정말 토크가 높을까? 공정 조건 B는 정말 더 나을까?

| 표시 | 의미 |
|---|---|
| 📋 제공 | 코드가 들어 있습니다. 실행만 하세요 |
| ✍️ 입력 | 설명을 보고 직접 입력하세요 |
| 💬 해석 | 결과를 보고 답을 적어 보세요 |

## 0. 준비 📋

In [1]:
import pandas as pd
from scipy import stats

df = pd.read_parquet("../../data/factory_table.parquet")

# 공정 조건 A/B 실험: 같은 기간에 LOT을 무작위로 A·B에 배정
# roughness = 표면 거칠기(작을수록 좋음), defect = 불량 여부(1 = 불량)
ab = pd.read_csv("../../data/ab_test.csv")

## 1. 고장 vs 정상: 토크 평균 ✍️

먼저 두 그룹의 평균을 봅니다.

In [2]:
# 힌트: df.groupby("그룹")["값"].mean()
df.groupby('failure')['torque'].mean()

failure
0    39.629655
1    50.168142
Name: torque, dtype: float64

## 2. 독립 2표본 t-test ✍️

두 그룹의 산포가 다를 수 있으므로 `equal_var=False`(Welch t-test)를 씁니다.

> 변수 이름: `normal`, `fail`

In [3]:
# 힌트: stats.ttest_ind(그룹1, 그룹2, equal_var=False)
normal = df[ df['failure'] == 0 ]['torque'] # 정상 : 0
fail = df[ df['failure'] == 1 ]['torque']   # 고장 : 1

result = stats.ttest_ind(fail, normal, equal_var=False)
print( result )
print(f"평균 차이 {fail.mean() - normal.mean():.1f} Nm, t = {result.statistic}, p-value = {result.pvalue:.2e}")
# p-value 가 0.05 보다 작으면 차이가 있다.

TtestResult(statistic=11.780812727929126, pvalue=3.635738917439069e-27, df=345.98234465365084)
평균 차이 10.5 Nm, t = 11.780812727929126, p-value = 3.64e-27


**Q1.** 귀무가설은 무엇이고, 결과를 보고 어떻게 판단하나요? 보고서에는 어떻게 쓰면 좋을까요?

> 답:


## 3. A/B 테스트: 두 지표 한눈에 ✍️

In [4]:
ab.groupby('condition')[['roughness','defect']].mean()

,roughness,defect
condition,,
A,3.17472,0.044
B,3.09352,0.032


## 4. 평균 비교: 표면 거칠기 ✍️

거칠기는 연속값이므로 t-test로 비교합니다.

> 변수 이름: `a`, `b`

In [5]:
a = ab[ ab['condition'] == "A" ]
b = ab[ ab['condition'] == "B" ]
stats.ttest_ind(a['roughness'], b['roughness'])

TtestResult(statistic=3.1546094202602655, pvalue=0.001655199828160141, df=998.0)

## 5. 비율 비교: 불량률 ✍️

불량 여부는 0/1이므로 **카이제곱 검정**으로 비율을 비교합니다. `pd.crosstab`으로 조건 × 불량 건수 표를 먼저 만듭니다.

> 변수 이름: `table`

In [8]:
# 힌트: pd.crosstab(행, 열), stats.chi2_contingency(표)
table = pd.crosstab( ab['condition'], ab['defect'] )
display(table)
result = stats.chi2_contingency(table)
print(f"p-value = {result.pvalue:.3f}")

defect,0,1
condition,,
A,478,22
B,484,16


p-value = 0.408


## 정리 💬

**Q2.** 거칠기와 불량률, 각각 어떤 결론을 내릴 수 있나요?

> 답:


**Q3.** 불량률 차이가 유의하지 않다는 것은 "B와 A의 불량률이 같다"는 뜻일까요? 현장에 무엇을 제안하겠습니까?

> 답:
